Shapely is a Python library used for the manipulation and analysis of 2D planar geometric objects (Points, LineStrings, Polygons) using the GEOS library. Key uses include creating geometries, calculating spatial attributes (area, length, distance), and performing topological operations (union, intersection, buffer). It is essential for GIS, spatial data analysis, and CAD-like tasks, offering efficient handling of vector data, typically integrated with libraries like GeoPandas or NumPy. 

In [2]:
pip install shapely

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
from shapely.geometry import Point, LineString, Polygon

In [4]:
p = Point(2, 3)
print(p)
print(p.x, p.y)

POINT (2 3)
2.0 3.0


In [5]:
line = LineString([(0, 0), (2, 2), (4, 1)])
print(line.length)

5.06449510224598


In [6]:
poly = Polygon([(0, 0), (4, 0), (4, 4), (0, 4)])
print(poly.area)

16.0


In [7]:
p1 = Point(0, 0)
p2 = Point(3, 4)

print(p1.distance(p2))  # Output: 5.0

5.0


In [8]:
buffer = p1.buffer(2)
print(buffer.area)

12.546193962183759


In [9]:
poly1 = Polygon([(0,0),(4,0),(4,4),(0,4)])
poly2 = Polygon([(2,2),(6,2),(6,6),(2,6)])

inter = poly1.intersection(poly2)
print(inter.area)

4.0


In [10]:
pt = Point(1, 1)
print(poly1.contains(pt))

True


In [11]:
pole = Point(5, 5)
boundary = Polygon([(0,0),(10,0),(10,10),(0,10)])

if boundary.contains(pole):
    print("Pole is inside service area")

Pole is inside service area


In [12]:
from shapely.geometry import Point, Polygon

p = Point(3, 3)

square = Polygon([
    (0, 0),
    (5, 0),
    (5, 5),
    (0, 5)
])

print(square.contains(p))
print(square.distance(p))

True
0.0


In [13]:
from shapely.geometry import Point, Polygon

poly = Polygon([(0,0),(4,0),(4,4),(0,4)])

p1 = Point(2, 2)   # inside
p2 = Point(4, 2)   # on boundary
p3 = Point(6, 2)   # outside

print(poly.contains(p1))
print(poly.touches(p2))
print(poly.intersects(p3))

True
True
False


In [14]:
from shapely.geometry import LineString, Polygon

boundary = Polygon([(0,0),(6,0),(6,6),(0,6)])
line = LineString([(-1,3),(7,3)])

print(line.intersects(boundary))
print(line.within(boundary))

True
False


In [15]:
from shapely.geometry import Point, LineString

pole = Point(5, 5)
buffer_zone = pole.buffer(2)

line = LineString([(1,5),(10,5)])

print(line.intersects(buffer_zone))

True


In [17]:
from shapely.geometry import Polygon

geom = Polygon([(0,0), (4,0), (4,4), (0,4)])

# Fix invalid geometry (if any)
clean_geom = geom.buffer(0)

print(clean_geom.is_valid)

True


In [18]:
from shapely.geometry import Point, Polygon

# Service boundary
service_area = Polygon([
    (0, 0),
    (10, 0),
    (10, 10),
    (0, 10)
])

# Poles
poles = [
    Point(2, 3),
    Point(8, 8),
    Point(12, 5),  # outside
    Point(5, 11),  # outside
    Point(4, 4)
]

# TODO: find poles outside the service area
outside_poles = []

for pole in poles:
    # your logic here
    pass

print("Outside poles:", outside_poles)


Outside poles: []


In [19]:
from shapely.geometry import Point, Polygon

# Service boundary
service_area = Polygon([
    (0, 0),
    (10, 0),
    (10, 10),
    (0, 10)
])

# Poles
poles = [
    Point(2, 3),
    Point(8, 8),
    Point(12, 5),  # outside
    Point(5, 11),  # outside
    Point(4, 4)
]

outside_poles = []

for pole in poles:
    if not service_area.contains(pole):
        outside_poles.append(pole)

print("Outside poles:", outside_poles)


Outside poles: [<POINT (12 5)>, <POINT (5 11)>]


In [22]:
from shapely.geometry import Point, LineString

# Transformer location
transformer = Point(5, 5)

# 2 meter safety buffer
restricted_zone = transformer.buffer(2)

# Power lines
lines = [
    LineString([(1, 1), (2, 2)]),       # safe
    LineString([(1, 5), (10, 5)]),      # violates
    LineString([(8, 8), (9, 9)]),       # safe
    LineString([(3, 5), (7, 5)])        # violates
]

violations = []

for i, line in enumerate(lines, start=1):
    if line.intersects(restricted_zone):
        print(f"Line {i} violates safety buffer")
        violations.append(i)

print("Violating line numbers:", violations)


Line 2 violates safety buffer
Line 4 violates safety buffer
Violating line numbers: [2, 4]


In [24]:
from shapely.geometry import Point, LineString
from shapely.strtree import STRtree

# Create buffers
points = [Point(5,5), Point(15,15), Point(25,25)]
buffers = [p.buffer(2) for p in points]

# Build index
tree = STRtree(buffers)

# Lines
lines = [
    LineString([(1,5),(10,5)]),    # hits buffer 1
    LineString([(14,15),(20,15)]), # hits buffer 2
    LineString([(50,50),(60,60)])  # safe
]

for i, line in enumerate(lines, 1):
    candidate_indices = tree.query(line)

    for idx in candidate_indices:
        if line.intersects(buffers[idx]):
            print(f"Line {i} violates buffer")


Line 1 violates buffer
Line 2 violates buffer
